# Reusable template — unit-string roster → exam / assessment score
## Clone of `ExamPred`


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

DATA_PATH = "data/student_exams.csv"
TARGET = "exam_score"
HOUR_COLS = ["study_hours","prep_hours","sleep_hours"]
GPA_COL, PCT_COL = "prior_gpa", "attendance"
CARD_CUTOFF, TEST_SIZE, RANDOM_STATE = 5, 0.2, 42
DROP_LEAK = ["effort_stars"]  # set [] to keep the post-hoc band


In [ ]:
df = pd.read_csv(DATA_PATH)
for col in HOUR_COLS:
    df[col] = df[col].astype(str).str.replace(" hours","",regex=False).astype(float)
df["study_load"] = df["study_hours"] + df["prep_hours"]
df[GPA_COL] = df[GPA_COL].astype(str).str.replace(" GPA","",regex=False).astype(float)
df[PCT_COL] = df[PCT_COL].astype(str).str.replace("%","",regex=False).astype(float) / 100
# add cohort / partners / '-week' / ' cr' strips here

y = df[TARGET]
X = df.drop(columns=[TARGET] + [c for c in DROP_LEAK if c in df.columns])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
cat = X_train.select_dtypes(include=["object","category"]).columns.tolist()
low = [c for c in cat if X[c].nunique() < CARD_CUTOFF]
high = [c for c in cat if X[c].nunique() >= CARD_CUTOFF]
for col in high:
    means = pd.concat([X_train[col], y_train], axis=1).groupby(col)[TARGET].mean()
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(y_train.mean())
for col in low:
    dtr = pd.get_dummies(X_train[col], prefix=col, drop_first=True)
    dte = pd.get_dummies(X_test[col], prefix=col, drop_first=True)
    dte = dte.reindex(columns=dtr.columns, fill_value=0)
    X_train = pd.concat([X_train.drop(columns=[col]), dtr], axis=1)
    X_test = pd.concat([X_test.drop(columns=[col]), dte], axis=1)
obj = X_train.select_dtypes(include=["object"]).columns.tolist()
X_train, X_test = X_train.drop(columns=obj), X_test.drop(columns=obj)
model = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
pred = model.predict(X_test)
print("MAE", mean_absolute_error(y_test, pred))
print("R2 ", r2_score(y_test, pred))
print("baseline MAE", mean_absolute_error(y_test, np.full_like(y_test, y_train.mean(), dtype=float)))
print(pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(8))
